# Adaptive QA: Corrective Diagnostic (Training-Only Corpus + 0.5B vs. 3B Ablation) & Stage 4 Controller Check

This notebook executes all corrective steps requested by Dr. Ahmadnia while preserving your original Stage 2 and Stage 3 runs untouched:
1. **Training-Only Wikipedia Corpus (`evidence_corpus_qanta2021_train_top500.jsonl`):** Rebuilds and freezes a 500-article Wikipedia corpus selected **strictly from `qanta.train.2021.12.20.json`** (top-500 training topics by question frequency; zero access to `selection.json`, `pilot10.jsonl`, `pilot50.jsonl`, or `qanta.dev`).
2. **Prompt `qa-answer-v4` & `0.5B` vs. `3B` Diagnostic Comparison:** Runs the 10-question diagnostic check on the training-only corpus with **both** pinned `Qwen/Qwen2.5-0.5B-Instruct` and pinned `Qwen/Qwen2.5-3B-Instruct`, isolating prompt/retrieval effects from model size and eliminating generic `"this novel"` / bare-string outputs.
3. **Stage 4 Controller & Calibration Evaluation:** Runs the 5-fold question-grouped cross-validation controller and calibration checks and packages all outputs into **`STAGE_04_CORRECTIVE_RESULTS.zip`** for download.

In [ ]:
from google.colab import drive, files
from pathlib import Path, PurePosixPath
from io import BytesIO
import os, sys, json, zipfile, shutil, subprocess

drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/AdaptiveQA2021_consistency')
PREV_STAGE2 = BASE_DIR / 'adaptive-confidence-agent'
PREV_STAGE3 = BASE_DIR / 'adaptive-qa-stage03-v3'
PROJECT = BASE_DIR / 'adaptive-qa-stage04-corrective'
PROJECT.mkdir(parents=True, exist_ok=True)
print('Corrective workspace (preserves Stage 2 and Stage 3 untouched):', PROJECT)

In [ ]:
# Sync stage_04_corrective.zip from Drive (or upload if needed)
zip_cands = sorted(BASE_DIR.glob('stage_04_corrective*.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
if zip_cands:
    print('Loading source archive from Drive:', zip_cands[0])
    raw_bytes = zip_cands[0].read_bytes()
else:
    uploaded = files.upload()
    raw_bytes = next(iter(uploaded.values()))

with zipfile.ZipFile(BytesIO(raw_bytes)) as zf:
    for entry in zf.infolist():
        if entry.is_dir():
            continue
        parts = PurePosixPath(entry.filename).parts
        rel = PurePosixPath(*parts[1:]) if parts[0] == 'adaptive-qa' else PurePosixPath(*parts)
        target = PROJECT / str(rel)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(zf.read(entry.filename))

os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[gpu,data]'], check=True)
print('Installed package in:', PROJECT)

In [ ]:
# Copy frozen QANTA 2021 raw training file and reviewed pilot10/pilot50 splits from Stage 3/2
for donor in (PREV_STAGE3, PREV_STAGE2):
    if (donor / 'data/raw/qanta2021/qanta.train.2021.12.20.json').exists() and not (PROJECT / 'data/raw/qanta2021/qanta.train.2021.12.20.json').exists():
        shutil.copytree(donor / 'data/raw/qanta2021', PROJECT / 'data/raw/qanta2021', dirs_exist_ok=True)
        print('Copied raw QANTA 2021 files from:', donor)
    if (donor / 'data/processed/qanta2021/frozen_manifest.json').exists() and not (PROJECT / 'data/processed/qanta2021/frozen_manifest.json').exists():
        shutil.copytree(donor / 'data/processed/qanta2021', PROJECT / 'data/processed/qanta2021', dirs_exist_ok=True)
        print('Copied frozen reviewed QANTA 2021 splits from:', donor)

if not (PROJECT / 'data/raw/qanta2021/qanta.train.2021.12.20.json').exists():
    subprocess.run([sys.executable, 'scripts/prepare_qanta_public.py', 'download'], cwd=PROJECT, check=True)

# Run all 29 unit tests (including the strict training-only corpus test)
res = subprocess.run([sys.executable, 'scripts/run_tests.py'], cwd=PROJECT, capture_output=True, text=True)
test_log = (res.stdout or '') + (res.stderr or '')
print(test_log)
assert res.returncode == 0, 'Unit tests failed!'
(PROJECT / 'results').mkdir(parents=True, exist_ok=True)
(PROJECT / 'results/stage04_tests.txt').write_text(test_log, encoding='utf-8')

## 1. Rebuild & Freeze Strictly Training-Only Wikipedia Corpus (`top-500` QANTA training topics)

This step runs `scripts/build_wikipedia_corpus.py`, which reads **only `qanta.train.2021.12.20.json`** (`fold == 'train'`) and selects the top 500 training topics by question frequency without touching `selection.json`, `pilot10.jsonl`, `pilot50.jsonl`, or `qanta.dev.2021.12.20.json`.

In [ ]:
subprocess.run([
    sys.executable, '-u', 'scripts/build_wikipedia_corpus.py',
    '--contact', 'https://qanta-org.github.io',
    '--limit', '500',
    '--overwrite-corpus'
], cwd=PROJECT, check=True)

## 2. Run 10-Question Diagnostic with `Qwen2.5-0.5B-Instruct` vs. `Qwen2.5-3B-Instruct` on Training-Only Corpus

This step runs the 10-question diagnostic check on the new training-only corpus first with pinned `Qwen/Qwen2.5-0.5B-Instruct` and then with pinned `Qwen/Qwen2.5-3B-Instruct` under `qa-answer-v4`, isolating the effect of prompt/retrieval changes from the model upgrade.

In [ ]:
CFG_05B = 'configs/stage04_diag10_qwen05b_traincorpus.yaml'
CFG_3B  = 'configs/stage04_diag10_qwen3b_traincorpus.yaml'

for cfg in (CFG_05B, CFG_3B):
    print('=== Running diagnostic config:', cfg, '===')
    subprocess.run([sys.executable, '-u', 'scripts/run.py', 'prepare', '--config', cfg], cwd=PROJECT, check=True)
    subprocess.run([sys.executable, '-u', 'scripts/run.py', 'baseline', '--config', cfg, '--execute-model'], cwd=PROJECT, check=True)
    subprocess.run([sys.executable, '-u', 'scripts/run.py', 'collect', '--config', cfg, '--execute-model'], cwd=PROJECT, check=True)
    subprocess.run([sys.executable, '-u', 'scripts/verify_collection.py', '--config', cfg], cwd=PROJECT, check=True)
    subprocess.run([sys.executable, '-u', 'scripts/run.py', 'pilot-report', '--config', cfg], cwd=PROJECT, check=True)

## 3. Run Stage 4 Controller & Calibration Evaluation and Package Results ZIP

In [ ]:
# Copy completed Stage 3 pilot50 run into results/runs so Stage 4 controller evaluation also runs
s3_run = PREV_STAGE3 / 'results/runs/qanta2021_stage03_pilot50_qwen3b_v3'
target_s3_run = PROJECT / 'results/runs/qanta2021_stage03_pilot50_qwen3b_v3'
if s3_run.is_dir() and not target_s3_run.exists():
    shutil.copytree(s3_run, target_s3_run, dirs_exist_ok=True)
    shutil.copy2(PREV_STAGE3 / 'data/raw/evidence_corpus_qanta2021.jsonl', PROJECT / 'data/raw/evidence_corpus_qanta2021.jsonl')

subprocess.run([sys.executable, '-u', 'scripts/run.py', 'evaluate', '--config', 'configs/stage03_pilot50.yaml'], cwd=PROJECT, check=True)

out_zip = Path('/content/STAGE_04_CORRECTIVE_RESULTS.zip')
with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(PROJECT / 'results/stage04_tests.txt', 'stage04_tests.txt')
    zf.write(PROJECT / 'data/raw/wikipedia_train_top500/corpus_manifest.json', 'train_top500_corpus_manifest.json')
    zf.write(PROJECT / 'data/raw/wikipedia_train_top500/title_selection.json', 'train_top500_title_selection.json')
    for run_id in ('qanta2021_stage04_diag10_qwen05b_traincorpus', 'qanta2021_stage04_diag10_qwen3b_traincorpus'):
        rdir = PROJECT / 'results/runs' / run_id
        for fn in ('pilot_report.json', 'pilot_report.md', 'collection_verification.json', 'manifest.json'):
            if (rdir / fn).is_file():
                zf.write(rdir / fn, f'{run_id}/{fn}')
        for fn in ('pilot_by_prefix.csv', 'sample_review.csv', 'retrieval_review.csv'):
            if (rdir / 'tables' / fn).is_file():
                zf.write(rdir / 'tables' / fn, f'{run_id}/tables/{fn}')
    if target_s3_run.is_dir():
        for fn in ('stage4_calibration_report.json', 'stage5_controller_evaluation.json'):
            if (target_s3_run / fn).is_file():
                zf.write(target_s3_run / fn, f'stage4_controller/{fn}')
        for sub in ('tables', 'figures'):
            for p in sorted((target_s3_run / sub).glob('*')):
                zf.write(p, f'stage4_controller/{sub}/{p.name}')

print('Created:', out_zip, 'bytes:', out_zip.stat().st_size)
files.download(str(out_zip))